# Privacy & Data Protection— Privacy as Measurable Properties

**Module 15 · Lesson 15.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- PII Detection & Classification
- Re-identification: Anonymous But Unique
- k-Anonymity: Hide In A Crowd Of k
- l-Diversity: k Is Not Enough
- Differential Privacy: Noise You Can Trust
- Membership Inference

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## “We removed the names” is not privacy

> 💡 **Info**
>
> The most expensive privacy failures in AI are not exotic — they come from treating privacy as a checkbox instead of a measurable property. A dataset with the names stripped that a researcher re-identifies from ZIP, birth date, and sex. A model that repeats a customer’s support email verbatim when prompted just so. A trace store that quietly became a PII store because someone logged raw prompts “for debugging.” An erasure request that cannot be honored because the person’s data is baked into a trained model. Each of these is discoverable and preventable *before* launch with a small amount of code and discipline. This lesson treats privacy the way you already treat reliability: as properties you can measure and enforce. You will classify the personal data into direct and quasi-identifiers, measure how re-identifiable each record is, climb the anonymization ladder from k-anonymity to l-diversity, reach the one guarantee that survives an attacker with unlimited side data — differential privacy — test whether your trained model leaks who was in it, and gate the pipeline so raw PII is never logged and a later erasure request stays possible. None of it makes you a privacy lawyer; it makes you the engineer who can say, with numbers, how private the system actually is.

### 🎯 What you will be able to do after this lesson

- **Apply** PII classification and a re-identification test to show that removing direct identifiers does not make data anonymous.

- **Analyze** a dataset with k-anonymity and l-diversity, and explain the homogeneity attack that k alone does not stop.

- **Evaluate** a differentially private release by its epsilon budget, and a trained model for membership-inference leakage.

- **Create** a privacy pipeline gate (minimize, redact, do-not-log, encrypt, honor erasure) and reason about why unlearning is hard.

> 📦 **Info**
>
> ✅ Before you startThis lesson follows **15.3 (Copyright & IP)**: copyright is about the *rights* in the training data; privacy is about the *person* in it — their identity and their secrets. The same document can be both a copyright and a privacy problem. It also builds on **Module 08** (RAG retrieves documents into the prompt — and if those documents hold customer PII, it flows into the model call and the logs) and **Module 14** (an observability trace records the prompt and response — a trace store becomes a PII store unless you redact before the span). Everything here is keyless arithmetic on one illustrative dataset.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🔎 **Analogy**
>
> PII detection is a **metal detector at the door**. Before anyone walks into the secure area, they pass through a scanner that beeps at metal — not because metal is evil, but because you need to *know it is there* to decide what to do about it. A privacy pipeline needs the same first step: scan every field and flag the personal data before it flows anywhere. And like a good detector, it has to distinguish the obvious from the subtle — a name badge (a direct identifier) sets it off loudly, but so should the combination of small items that together give someone away (the quasi-identifiers). **Where it breaks down:** a metal detector only finds metal it is tuned for; a PII scanner only finds patterns it knows, so novel or free-text PII can slip through, which is why minimization matters more than detection alone.

> 📦 **Info**
>
> 🚫 Three misconceptions this lesson kills
> - **“We removed the names, so it is anonymous.”** Quasi-identifiers (ZIP, birth date, sex) re-identify about 87 percent of people in combination. Deleting direct identifiers is necessary but not sufficient.
> - **“k-anonymity makes it safe.”** A k-anonymous group whose members all share one sensitive value still leaks it — the homogeneity attack. You need l-diversity too.
> - **“The model does not output raw data, so it is private.”** Membership inference reads the loss to tell who was in the training set, no raw output required.

> 💡 **Info**
>
> ⚠️ Anti-patternLogging raw prompts and responses “for debugging” while claiming the data is protected. It is the most common real privacy leak: every trace store, analytics pipeline, and log aggregator quietly becomes an unmanaged copy of your users’ personal data, often in systems with looser access controls than the primary database. Worse, it makes a later erasure request impossible to honor — you cannot delete what you cannot find. Every step below builds toward the opposite discipline: redact before the model call, redact before the log, and never let raw PII come to rest where you did not intend.

---

## 🎯 Concept 1: PII Detection & Classification

### PII Detection & Classification

Privacy work starts by finding the personal data and sorting it. DIRECT identifiers (name, email, SSN) name a person on their own; QUASI-identifiers (ZIP, birth date, sex) name them only in COMBINATION; a SENSITIVE attribute (a diagnosis, a salary) is what an attacker wants to learn. Dropping the direct identifiers is necessary - but, as the next step shows, not sufficient.

You cannot protect what you have not found, so privacy begins with a scan and a classification. Personal data is not one thing. **Direct identifiers** — a name, an email, a national ID — pick out a person all by themselves. **Quasi-identifiers** — a ZIP code, a birth date, a sex, a job title — look harmless individually but combine into a near-unique fingerprint. And the **sensitive attribute** — a diagnosis, a salary, a religion — is the thing an attacker is trying to learn about a specific person. In the worked example, an eight-record clinic export has two direct identifiers (name, email), three quasi-identifiers (ZIP, birth, sex), and one sensitive attribute (diagnosis), and all eight records carry a direct identifier. The instinct is to delete the name and email and call it done — and that is necessary, but the next step shows why it is nowhere near sufficient. The block classifies the fields and counts the exposure, keyless.

> 🪛 **Analogy**
>
> The three field classes are like the difference between **a passport, a jigsaw piece, and the secret it guards**. A passport (a direct identifier) names you outright. A single jigsaw piece (a quasi-identifier) — your postcode, say — looks like nothing on its own, but snap enough pieces together and a picture of exactly one person emerges. And the secret (the sensitive attribute) is what the whole exercise is meant to keep — your diagnosis, your salary. Privacy engineering is about knowing which fields are passports, which are jigsaw pieces, and which are the secret, because you protect each differently: you remove passports, you blur jigsaw pieces, and you guard the secret hardest of all.

You delete the name and email columns from a dataset. The ZIP, birth date, and sex remain. Is the dataset now anonymous?

**📝 `01_pii_detection.py`** - *runnable*

In [ ]:
# PII DETECTION & CLASSIFICATION: privacy work starts by finding the personal data. Fields split into DIRECT identifiers
# (identify a person alone - name, email, SSN) and QUASI-identifiers (identify only in COMBINATION - ZIP, birth date, sex).
rows = [  # (name, email, zip, birth, sex, diagnosis) - an illustrative clinic export
    ("Ada",   "ada@x.com",   "02138", "1985-03-11", "F", "flu"),
    ("Bao",   "bao@x.com",   "02139", "1985-07-22", "M", "flu"),
    ("Chen",  "chen@x.com",  "02141", "1985-09-02", "F", "flu"),
    ("Diego", "diego@x.com", "02142", "1985-01-30", "M", "flu"),
    ("Eve",   "eve@x.com",   "10001", "1972-06-15", "F", "diabetes"),
    ("Farah", "farah@x.com", "10002", "1972-11-03", "F", "cancer"),
    ("Gus",   "gus@x.com",   "10003", "1972-02-19", "M", "asthma"),
    ("Hana",  "hana@x.com",  "10004", "1972-08-27", "F", "flu")]
fields = ["name", "email", "zip", "birth", "sex", "diagnosis"]
DIRECT = {"name", "email"}            # each identifies a person on its own
QUASI = {"zip", "birth", "sex"}       # identify a person only in combination
SENSITIVE = {"diagnosis"}             # what an attacker wants to learn
print("{} records, {} fields.".format(len(rows), len(fields)))
print("  direct identifiers   : {}".format(sorted(DIRECT)))
print("  quasi-identifiers    : {}".format(sorted(QUASI)))
print("  sensitive attribute  : {}".format(sorted(SENSITIVE)))
with_direct = sum(1 for r in rows if r[0] or r[1])
print()
print("{}/{} records carry a DIRECT identifier - dropping name+email is necessary but NOT sufficient.".format(with_direct, len(rows)))
print("The quasi-identifiers still leak: the next step shows they can re-identify a person all by themselves.")

# Output:
# 8 records, 6 fields.
#   direct identifiers   : ['email', 'name']
#   quasi-identifiers    : ['birth', 'sex', 'zip']
#   sensitive attribute  : ['diagnosis']
#
# 8/8 records carry a DIRECT identifier - dropping name+email is necessary but NOT sufficient.
# The quasi-identifiers still leak: the next step shows they can re-identify a person all by themselves.

- Fields sort into **direct** identifiers (name, email), **quasi**-identifiers (ZIP, birth, sex), and a **sensitive** attribute (diagnosis).
- All 8 records carry a direct identifier — the obvious PII to remove.
- But removing name and email is **necessary, not sufficient**: the quasi-identifiers remain.
- Those quasi-identifiers are the leak the next step measures — they re-identify people in combination.

#### 💡 What just happened

⚡ What just happened?Privacy starts by classifying fields into direct identifiers (name you alone), quasi-identifiers (name you in combination), and the sensitive attribute (what an attacker wants). Tradeoff: detection finds known patterns; novel free-text PII can slip through, so minimization beats detection alone. Next: why deleting the direct identifiers does not make the data anonymous.

- A table whose columns light up as direct / quasi / sensitive
- 8 records; 2 direct, 3 quasi, 1 sensitive

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Re-identification: Anonymous But Unique

### Re-identification: Anonymous But Unique

Strip the names and the data still is not anonymous. A landmark result (Latanya Sweeney) is that (5-digit ZIP + full birth date + sex) uniquely identifies about 87 percent of the US population - she re-identified a governor’s medical record by cross-linking an “anonymous” hospital release to a public voter roll. A record unique on its quasi-identifiers is directly re-identifiable.

Here is the result that reshaped privacy law. In the late 1990s, Latanya Sweeney took a hospital dataset released as “anonymous” — names removed, but ZIP, birth date, and sex intact — bought a public voter roll for the same town, joined the two on those three quasi-identifiers, and re-identified the medical record of the state’s governor. She then showed the general result: the combination (5-digit ZIP + full birth date + sex) is *unique* for about **87 percent** of Americans. The lesson is that anonymity is not a property of one dataset in isolation — it is a property of that dataset *plus every other dataset an attacker can link to it*, and there are a great many linkable public datasets. In the worked example, all eight records are unique on their quasi-identifiers, so every one is directly re-identifiable by cross-linking. This is why “we removed the names” is not a privacy control, and it is the motivation for everything that follows: to be anonymous, a record has to stop being unique. The block counts the unique records and states the re-identification rate, keyless.

> 👤 **Analogy**
>
> Re-identification is a **dating profile with no name that still describes exactly one person**. “A 43-year-old left-handed trombonist in this postcode who breeds corgis” names nobody directly — and yet, in a town, it is unmistakably one specific human. Each detail narrows the field; a few specific details together leave a single match. Quasi-identifiers work the same way: your ZIP thins the crowd to a neighborhood, your exact birth date to a handful, your sex to one. No name is needed because the *combination* is the name. Anonymity is not about hiding the label; it is about making sure enough people fit the description that no single person is picked out.

A hospital releases records with names removed but ZIP, birth date, and sex kept. How re-identifiable is it?

**📝 `02_re_identification.py`** - *runnable*

In [ ]:
# RE-IDENTIFICATION VIA QUASI-IDENTIFIERS: strip the names and the data still is not anonymous. A famous result (Sweeney)
# is that {5-digit ZIP, full birth date, sex} uniquely identifies ~87% of the US population. Count how many rows are unique.
rows = [  # (zip, birth, sex) - the quasi-identifiers that survive after name+email are removed
    ("02138", "1985-03-11", "F"), ("02139", "1985-07-22", "M"), ("02141", "1985-09-02", "F"),
    ("02142", "1985-01-30", "M"), ("10001", "1972-06-15", "F"), ("10002", "1972-11-03", "F"),
    ("10003", "1972-02-19", "M"), ("10004", "1972-08-27", "F")]
counts = {}
for q in rows:
    counts[q] = counts.get(q, 0) + 1
unique = [q for q in rows if counts[q] == 1]
print("Names removed. Re-identification risk from the quasi-identifiers (zip, birth, sex):")
print("  {} of {} records are UNIQUE on their quasi-identifiers -> {:.0%} directly re-identifiable.".format(len(unique), len(rows), len(unique) / len(rows)))
print()
print("Anonymity is not just 'delete the name'. Sweeney showed {ZIP, birth date, sex} pins down ~87% of Americans by cross-linking")
print("to a public voter roll. The fix is to make each record blend into a crowd - which is what k-anonymity measures next.")

# Output:
# Names removed. Re-identification risk from the quasi-identifiers (zip, birth, sex):
#   8 of 8 records are UNIQUE on their quasi-identifiers -> 100% directly re-identifiable.
#
# Anonymity is not just 'delete the name'. Sweeney showed {ZIP, birth date, sex} pins down ~87% of Americans by cross-linking
# to a public voter roll. The fix is to make each record blend into a crowd - which is what k-anonymity measures next.

- With names removed, each record is shown by its quasi-identifiers (ZIP, birth, sex) only.
- All **8 of 8** records are unique on those quasi-identifiers — 100 percent directly re-identifiable.
- The attack is a **cross-link**: join the “anonymous” table to a public voter roll on the shared quasi-identifiers.
- Sweeney: (ZIP + birth date + sex) pins down about 87 percent of the US population — anonymity is not “delete the name.”

#### 💡 What just happened

⚡ What just happened?Anonymity is a property of a dataset plus everything an attacker can link to it: records unique on their quasi-identifiers are re-identifiable, and (ZIP + birth date + sex) pins down ~87 percent of people. Tradeoff: none - it is a threat to defend. Next: the fix is to make each record stop being unique - to hide in a crowd.

- Quasi-identifier rows shown unique, then cross-linked to a voter roll
- 8/8 unique → 100 percent re-identifiable; Sweeney ~87 percent

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: k-Anonymity: Hide In A Crowd Of k

### k-Anonymity: Hide In A Crowd Of k

A table is k-anonymous if every record shares its quasi-identifiers with at least k-1 others - so no one is unique. You raise k by GENERALIZING (ZIP 02138 to 02, a birth year to a decade) and SUPPRESSING (dropping a column). Bigger k is more private but coarser data - a genuine privacy-versus-utility tradeoff.

If uniqueness is the problem, the fix is to make each record blend in. A table is **k-anonymous** if, for the quasi-identifiers, every record is identical to at least **k-1 others** — so any attacker who narrows down to a record narrows down to a group of at least k people, not one. You raise k with two operations. **Generalization** makes values coarser: a 5-digit ZIP becomes its first two digits, a full birth date becomes a decade, a precise age becomes a 10-year band. **Suppression** drops a value or a whole column entirely. In the worked example the raw table has **k=1** (every record unique). Generalize the ZIP to two digits, the birth to a decade, and suppress sex, and the eight records collapse into two groups of four — **k=4**. Now each person hides among four identical ones. The catch is the *cost*: the data got coarser, and a query that needed the exact ZIP or age can no longer be answered. This is the fundamental privacy-utility tradeoff, and k is the dial — bigger k, more privacy, less useful data. But k alone has a hole, which the next step exposes. The block computes k before and after generalization, keyless.

> 👥 **Analogy**
>
> k-anonymity is **wearing the same uniform as everyone in the crowd**. If you are the only person in a bright red coat, a witness picks you out instantly (k=1). Put four people in identical red coats and the witness can only say “one of those four” — you are hidden in a crowd of k=4. Generalizing the data is how you hand out the matching uniforms: blur everyone’s exact age into a decade and their exact ZIP into a region until enough people look alike. The tradeoff is obvious the moment you try it: the more identical you make everyone, the less you can tell them apart — which is exactly what privacy wants and what data analysis loses.

A raw table has k=1. You generalize ZIP to 2 digits and birth to a decade and drop sex, giving two groups of four. What did you achieve, and at what cost?

**📝 `03_k_anonymity.py`** - *runnable*

In [ ]:
# k-ANONYMITY: a table is k-anonymous if every record shares its quasi-identifiers with at least k-1 others - so no one is
# unique. You raise k by GENERALIZING (zip 02138 -> 02, birth 1985 -> the 1980s) and SUPPRESSING (drop sex). Measure k.
raw = [("02138", "1985", "F"), ("02139", "1985", "M"), ("02141", "1985", "F"), ("02142", "1985", "M"),
       ("10001", "1972", "F"), ("10002", "1972", "F"), ("10003", "1972", "M"), ("10004", "1972", "F")]
def k_of(table):
    groups = {}
    for q in table:
        groups[q] = groups.get(q, 0) + 1
    return min(groups.values()), groups
def generalize(z, y, s):
    return (z[:2], y[:3] + "0s")            # zip -> 2-digit region, year -> decade, sex SUPPRESSED (dropped)
k_raw, _ = k_of(raw)
gen = [generalize(*r) for r in raw]
k_gen, groups = k_of(gen)
print("raw quasi-identifiers (zip5, year, sex): k = {}  (every record is unique -> re-identifiable)".format(k_raw))
print("after generalize+suppress (zip2, decade, drop sex):")
for g, n in sorted(groups.items()):
    print("  {} -> {} records".format(g, n))
print("  k = {}  (each record now hides among {} identical ones)".format(k_gen, k_gen))
print()
print("k-anonymity trades detail for privacy: bigger k is more private but coarser data. But k alone is not enough -")
print("if everyone in a group shares the same secret, hiding WHICH person you are does not hide WHAT you have. See l-diversity.")

# Output:
# raw quasi-identifiers (zip5, year, sex): k = 1  (every record is unique -> re-identifiable)
# after generalize+suppress (zip2, decade, drop sex):
#   ('02', '1980s') -> 4 records
#   ('10', '1970s') -> 4 records
#   k = 4  (each record now hides among 4 identical ones)
#
# k-anonymity trades detail for privacy: bigger k is more private but coarser data. But k alone is not enough -
# if everyone in a group shares the same secret, hiding WHICH person you are does not hide WHAT you have. See l-diversity.

- **k** is the size of the smallest group sharing the same quasi-identifiers; raw table k=1 (all unique).
- **Generalize** (ZIP to 2 digits, birth to a decade) and **suppress** (drop sex) to merge records into groups.
- The eight records collapse into two groups of four — **k=4**; each record hides among four identical ones.
- The cost is coarser data: bigger k is more private but less useful — the privacy-utility tradeoff, with k as the dial.

#### 💡 What just happened

⚡ What just happened?k-anonymity makes every record share its quasi-identifiers with at least k-1 others (raw k=1 to k=4 here) by generalizing and suppressing. Tradeoff: bigger k is more private but coarser, less useful data. Next: k hides WHICH person you are, but if everyone in your group shares a secret, it does not hide WHAT you have.

- Rows generalize and merge into equivalence classes as k grows
- Raw k=1 → generalize+suppress → k=4 (two classes of 4)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: l-Diversity: k Is Not Enough

### l-Diversity: k Is Not Enough

A group can be k-anonymous and still leak. If all k members share the same SENSITIVE value, learning that someone is in the group reveals their value - even though you cannot tell which person they are. That is the homogeneity attack. l-diversity requires at least l distinct sensitive values per group; t-closeness goes further and matches the group’s distribution to the whole.

k-anonymity has a subtle and serious hole. It guarantees you cannot tell *which* of the k people a record belongs to — but it says nothing about *what* those k people have in common. If every member of a k-anonymous group shares the same sensitive value, then simply knowing someone is in that group discloses their value, no individual identification required. This is the **homogeneity attack**. In the worked example, the k=4 group for the 02 region in the 1980s has all four diagnoses equal to *flu* — so an attacker who knows a target is a 40-something in that region learns they have the flu, even though k-anonymity “protected” them. The other group, for the 10 region in the 1970s, has four *different* diagnoses, so membership reveals nothing. The fix is **l-diversity**: require each group to contain at least **l distinct sensitive values**. A stronger version, **t-closeness**, requires the group’s distribution of sensitive values to resemble the overall distribution, defeating attacks that exploit a skew even when l distinct values are present. The takeaway: k protects identity, l protects the secret, and you need both. The block checks l-diversity on each group and flags the homogeneous one, keyless.

> 🔑 **Analogy**
>
> l-diversity is the difference between **a crowd of strangers and a crowd leaving the same clinic**. Hiding in a crowd (k-anonymity) works only if the crowd is diverse. If you blend into four identical-looking people but *all four just walked out of the oncology ward*, then anyone who spots you in that group learns your secret without ever learning your name — the crowd itself gave it away. l-diversity insists the crowd be mixed: some from oncology, some from the pharmacy, some just visiting, so that being in the group tells an observer nothing about why you are there. A uniform crowd hides who you are and broadcasts what you have.

A group is k-anonymous (k=4), but all four members have the same diagnosis. Are those people protected?

**📝 `04_l_diversity.py`** - *runnable*

In [ ]:
# l-DIVERSITY (the k-anonymity homogeneity attack): a group can be k-anonymous yet still leak. If all k members share the
# same SENSITIVE value, learning someone is in the group reveals their value even though you cannot tell which person they are.
groups = {   # each is a k=4 equivalence class after generalization: (quasi-class -> list of sensitive values)
    "(02, 1980s)": ["flu", "flu", "flu", "flu"],                 # homogeneous
    "(10, 1970s)": ["diabetes", "cancer", "asthma", "flu"]}      # diverse
L_MIN = 2                                                        # require at least 2 distinct sensitive values per group
for cls, vals in groups.items():
    l = len(set(vals))
    ok = l >= L_MIN
    verdict = "OK (l-diverse)" if ok else "LEAKS - homogeneity attack (every member has the same value)"
    print("class {}: k=4, distinct sensitive values l={} -> {}".format(cls, l, verdict))
print()
print("Both groups are k-anonymous (k=4), but the first is l=1: anyone known to be a 40-something in the 02 region HAS the flu,")
print("no name needed. l-diversity requires variety in the sensitive column; t-closeness goes further and matches its distribution.")

# Output:
# class (02, 1980s): k=4, distinct sensitive values l=1 -> LEAKS - homogeneity attack (every member has the same value)
# class (10, 1970s): k=4, distinct sensitive values l=4 -> OK (l-diverse)
#
# Both groups are k-anonymous (k=4), but the first is l=1: anyone known to be a 40-something in the 02 region HAS the flu,
# no name needed. l-diversity requires variety in the sensitive column; t-closeness goes further and matches its distribution.

- Both groups are k-anonymous (k=4) — but k says nothing about the **sensitive** column.
- Group (02, 1980s) has all four diagnoses equal to flu — **l=1**, the homogeneity attack: membership reveals the diagnosis.
- Group (10, 1970s) has four distinct diagnoses — **l=4**, so membership reveals nothing.
- **l-diversity** requires at least l distinct sensitive values per group; **t-closeness** goes further and matches the distribution.

#### 💡 What just happened

⚡ What just happened?k-anonymity hides which person you are; l-diversity hides what you have. A k-anonymous group with one sensitive value (l=1) leaks it via the homogeneity attack, so require at least l distinct values per group (t-closeness matches the distribution). Tradeoff: more diversity means more generalization or suppression. Next: the guarantee that holds even against an attacker with unlimited side data.

- A k=4 class all sharing one diagnosis (l=1) leaks
- A diverse class (l=4) does not — the homogeneity attack

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Differential Privacy: Noise You Can Trust

### Differential Privacy: Noise You Can Trust

The strongest guarantee. Add calibrated noise so that one person joining or leaving changes any released answer by at most a factor of e^epsilon - so no output can be blamed on any individual. The Laplace mechanism uses noise scaled to sensitivity/epsilon. Epsilon is a privacy BUDGET: smaller epsilon means more noise and stronger privacy, and epsilons COMPOSE across queries.

k-anonymity and l-diversity protect a released *table*, but they can still fall to an attacker with enough outside data. **Differential privacy** (Dwork et al.) gives a guarantee that does not: it promises that whether or not *any single person* is in the dataset barely changes the probability of *any* released output. Formally, the odds of any result change by at most a factor of **e^epsilon**. You achieve it by adding calibrated random noise. The **Laplace mechanism** releases a numeric answer plus noise drawn at scale **sensitivity / epsilon**, where sensitivity is how much one person can change the true answer (for a count, 1). In the worked example, releasing a true count of 42 at **epsilon = 1.0** uses noise scale 1.0 and gives a weak-but-accurate guarantee (e^1.0 = 2.72); at **epsilon = 0.1** the scale jumps to 10 for a strong-but-noisy guarantee (e^0.1 = 1.11). Epsilon is a **privacy budget**: smaller is more private, and crucially it **composes** — two queries at epsilon 0.5 each spend epsilon 1.0 total, so you cannot escape the guarantee by asking many questions. This is the one privacy technique that holds against an adversary with unlimited side information, and **DP-SGD** applies the same idea to model training. The block runs the Laplace mechanism at two epsilons and shows the budget, keyless.

> 🎲 **Analogy**
>
> Differential privacy is **randomized response — the noisy survey that gives you deniability**. To ask a sensitive yes/no question, you tell each person: flip a coin secretly; on heads answer truthfully, on tails flip again and answer yes-or-no by that. Now anyone who answered “yes” can honestly say the coin made them — no single answer can be held against them — yet with enough responses you can still estimate the true rate by subtracting the known coin noise. Differential privacy is this idea made precise: add exactly enough calibrated noise that no individual’s presence changes the visible output, while the aggregate signal survives. Epsilon is how loaded the coin is — more noise, more deniability, less precision.

You release a count with the Laplace mechanism and lower epsilon from 1.0 to 0.1. What happens?

**📝 `05_differential_privacy.py`** - *runnable*

In [ ]:
# DIFFERENTIAL PRIVACY: the strongest guarantee. Add calibrated noise so one person joining or leaving barely changes any
# released answer - formally, the odds of any output change by at most e^epsilon. Smaller epsilon = more noise = more privacy.
import math
true_count = 42                                  # e.g. patients with a condition; hidden behind noise before release
sensitivity = 1                                  # one person changes the true count by at most 1
print("Releasing a count with the Laplace mechanism (noise ~ Laplace(0, sensitivity/epsilon)):")
for eps, drawn_noise in [(1.0, 1.3), (0.1, 8.0)]:  # drawn_noise = one illustrative fixed sample (real DP draws it randomly)
    scale = sensitivity / eps                    # bigger scale = more noise
    released = true_count + drawn_noise
    print("  epsilon={:.1f}: noise scale=sens/eps={:>4.1f}, privacy bound e^eps={:.2f}, released ~= {:.0f}".format(eps, scale, math.e ** eps, released))
print()
print("epsilon is a privacy BUDGET: e^0.1=1.11 (strong, noisy) vs e^1.0=2.72 (weak, accurate). And it COMPOSES -")
print("two queries at epsilon=0.5 each spend epsilon=1.0 total. DP is the one guarantee that survives an attacker with side data.")

# Output:
# Releasing a count with the Laplace mechanism (noise ~ Laplace(0, sensitivity/epsilon)):
#   epsilon=1.0: noise scale=sens/eps= 1.0, privacy bound e^eps=2.72, released ~= 43
#   epsilon=0.1: noise scale=sens/eps=10.0, privacy bound e^eps=1.11, released ~= 50
#
# epsilon is a privacy BUDGET: e^0.1=1.11 (strong, noisy) vs e^1.0=2.72 (weak, accurate). And it COMPOSES -
# two queries at epsilon=0.5 each spend epsilon=1.0 total. DP is the one guarantee that survives an attacker with side data.

- The **Laplace mechanism** releases a true count plus noise at scale **sensitivity / epsilon**.
- At epsilon 1.0: scale 1.0, guarantee e^eps = 2.72 (weaker privacy, more accurate).
- At epsilon 0.1: scale 10, guarantee e^eps = 1.11 (stronger privacy, noisier).
- Epsilon is a budget that **composes**: two queries at 0.5 each spend 1.0 — DP holds even against an attacker with unlimited side data.

#### 💡 What just happened

⚡ What just happened?Differential privacy adds noise at scale sensitivity/epsilon so no individual changes any output by more than a factor of e^epsilon; smaller epsilon is more private, and epsilons compose across queries. Tradeoff: stronger privacy (smaller epsilon) means less accuracy - the fundamental DP dial. Next: even without releasing a table, a trained model can leak who was in it.

- An epsilon slider widens a Laplace noise cloud over a true count of 42
- eps 1.0 → e^eps 2.72; eps 0.1 → e^eps 1.11; budget composes

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Membership Inference

### Membership Inference

Even a model that never emits raw data can leak WHO was in its training set. A model fits its training examples too well, so they get LOWER loss than unseen ones - and an attacker thresholds the loss to guess membership. A wide train/holdout loss gap is overfitting AND a privacy leak. Membership alone can be sensitive. Defend with regularization, early stopping, and DP-SGD.

The privacy story does not end at the dataset — the *model* leaks too. A **membership-inference attack** (Shokri et al.) asks a narrower but still-damaging question than “what was the data?”: *was this specific record in the training set?* The attack works because models overfit: they fit their training examples better than they fit unseen ones, so a training example tends to have **lower loss** (higher confidence) than a fresh one. An attacker who can measure the model’s loss on a record just thresholds it — low loss means “probably a member.” In the worked example, the five training records have losses from 0.10 to 0.30 and the five held-out records from 0.55 to 0.82, so a threshold at 0.5 separates them *perfectly*: a 100 percent successful attack. And membership alone can be deeply sensitive — “this person was in the HIV-study training set” leaks a medical fact without a single raw record being exposed. The defense is the same discipline that fixes overfitting: **regularization**, **early stopping**, and, for a formal guarantee, **DP-SGD** — training with differential privacy so the model cannot depend too much on any one example, which shrinks the loss gap toward the point where the attack is no better than a coin flip. The block runs the loss-threshold attack and reports its accuracy, keyless.

> 🎓 **Analogy**
>
> Membership inference is **a teacher who lights up at their own former students**. Show a professor a hundred faces and, without naming anyone, watch their reaction: the students they actually taught draw a flicker of recognition — a slightly warmer, faster response — while strangers get a blank look. You never learn a name, but from the *strength of the reaction* alone you can sort “taught them” from “never met them” with alarming accuracy. A model’s loss is that flicker of recognition: lower on the examples it was trained on. The fix is a teacher who reacts the same to everyone — which is exactly what regularization and DP-SGD enforce, by stopping the model from memorizing any individual too well.

Your model never outputs raw training data. Can an attacker tell whether a specific record was in its training set?

**📝 `06_membership_inference.py`** - *runnable*

In [ ]:
# MEMBERSHIP INFERENCE: even a model that never emits raw data can leak WHO was in its training set. A model fits its
# training examples too well, so they get LOWER loss than unseen ones - an attacker thresholds the loss to guess membership.
member_loss = [0.10, 0.18, 0.22, 0.15, 0.30]        # examples the model trained on -> memorized -> low loss
holdout_loss = [0.70, 0.55, 0.82, 0.60, 0.75]       # unseen examples -> higher loss
THRESH = 0.5                                          # attacker guesses "was a member" if loss < THRESH
tp = sum(1 for x in member_loss if x < THRESH)       # members correctly flagged
tn = sum(1 for x in holdout_loss if x >= THRESH)     # non-members correctly cleared
attack_acc = (tp + tn) / (len(member_loss) + len(holdout_loss))
print("Membership-inference attack (threshold the model's per-example loss):")
print("  members flagged   : {}/{}".format(tp, len(member_loss)))
print("  non-members cleared: {}/{}".format(tn, len(holdout_loss)))
print("  attack accuracy   : {:.0%}  (50% = the attacker learns nothing; higher = the model leaks membership)".format(attack_acc))
print()
print("A wide train/holdout loss gap is overfitting AND a privacy leak - membership alone can be sensitive ('was in the HIV study').")
print("The defenses are the same as for generalization: regularization, early stopping, and training with differential privacy (DP-SGD).")

# Output:
# Membership-inference attack (threshold the model's per-example loss):
#   members flagged   : 5/5
#   non-members cleared: 5/5
#   attack accuracy   : 100%  (50% = the attacker learns nothing; higher = the model leaks membership)
#
# A wide train/holdout loss gap is overfitting AND a privacy leak - membership alone can be sensitive ('was in the HIV study').
# The defenses are the same as for generalization: regularization, early stopping, and training with differential privacy (DP-SGD).

- The attacker thresholds the model’s per-example **loss**: low loss means “probably a training member.”
- Training records have low loss (0.10–0.30); held-out records have high loss (0.55–0.82).
- A threshold at 0.5 separates them perfectly — **100 percent** attack accuracy (50 percent would mean no leak).
- A wide train/holdout loss gap is overfitting **and** a privacy leak; defend with regularization, early stopping, and DP-SGD.

#### 💡 What just happened

⚡ What just happened?A model overfits its training data, so members get lower loss than non-members; thresholding the loss infers membership (100 percent here), and membership alone can be sensitive. Tradeoff: the defenses (regularization, DP-SGD) that close the leak can cost some accuracy. Next: turn all of this into a pipeline you can gate - and confront the hardest right, erasure.

- Member vs holdout loss piles split by a movable threshold
- Attack accuracy 100 percent at threshold 0.5; DP-SGD defends

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Privacy Pipeline & Right To Erasure

### The Privacy Pipeline & Right To Erasure

Privacy is an engineering gate, not a promise. Minimize what you collect, redact or pseudonymize before the model call, never log raw PII, encrypt, and honor deletion (GDPR Article 17). Erasing a person from a TRAINED model - “unlearning” - is hard, because retraining is costly. The practical answer is to minimize and pseudonymize up front so a later erasure request stays tractable.

Everything so far becomes a **gate** the feature must pass before it ships. The controls are concrete: **minimize** — collect only the personal data you actually need (GDPR Article 5); **redact or pseudonymize** PII before the model ever sees it and before anything is written to a log or trace; **never log raw** prompts or PII; **encrypt** in transit and at rest; and **honor erasure** — a data subject’s right to have their data deleted (GDPR Article 17, the “right to be forgotten”). In the worked example, four controls pass but one — “do not log raw prompts or PII” — is unmet, so the gate **BLOCKS** the release. And the hardest control is erasure, because deleting a person’s row from a database is trivial but **unlearning** them from a *trained model* is not: their influence is baked into the weights, and the only exact fix is often to retrain, which is expensive. Approaches like SISA training and approximate unlearning help, but the durable answer is upstream: **minimize and pseudonymize at ingestion** so you never trained on the raw person in the first place, which keeps a later erasure request tractable. Privacy by design is cheaper than privacy by retraining. The block evaluates the gate, keyless.

> 🎂 **Analogy**
>
> Erasing a person from a trained model is **trying to un-bake a cake to remove one egg**. Deleting their record from your database is like taking an egg out of the fridge — easy, it is a discrete thing sitting in one place. But once you have mixed that egg into the batter and baked it, the egg is *everywhere and nowhere*, distributed through the whole cake; there is no clean way to pick it back out short of baking a new cake from scratch (retraining). That is why privacy engineers obsess over what goes into the bowl: minimize and pseudonymize the ingredients up front, because the cheapest way to be able to remove an egg later is to have never fully baked it in.

A data subject requests erasure under GDPR Article 17, and your model was trained on their data. What is true?

**📝 `07_privacy_gate.py`** - *runnable*

In [ ]:
# THE PII PIPELINE + RIGHT TO ERASURE: privacy is an engineering gate, not a promise. Minimize what you collect, redact
# before the model sees it, never log raw PII, encrypt, and honor deletion (GDPR Art. 17 / CCPA). Erasing from a TRAINED model is hard.
CONTROLS = {                                          # each must hold for the pipeline to ship
    "collect only what is needed (data minimization)": True,
    "redact / pseudonymize PII before the model call": True,
    "do not log raw prompts or PII": False,           # UNMET - raw prompts are being logged
    "honor data-subject erasure (GDPR Art. 17)": True,
    "encrypt PII in transit and at rest": True}
unmet = [name for name, ok in CONTROLS.items() if not ok]
print("PRIVACY PIPELINE GATE (before a feature that touches personal data ships):")
for name, ok in CONTROLS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("gate: {}".format("PASS" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("Right-to-erasure is the hardest: deleting a row from a database is easy, but 'unlearning' a person from a trained model")
print("is not - retraining is costly. The practical answer is to minimize and pseudonymize UP FRONT so erasure stays tractable.")

# Output:
# PRIVACY PIPELINE GATE (before a feature that touches personal data ships):
#   [x] collect only what is needed (data minimization)
#   [x] redact / pseudonymize PII before the model call
#   [ ] do not log raw prompts or PII
#   [x] honor data-subject erasure (GDPR Art. 17)
#   [x] encrypt PII in transit and at rest
#
# gate: BLOCK - 1 unmet:
#    - do not log raw prompts or PII
# Right-to-erasure is the hardest: deleting a row from a database is easy, but 'unlearning' a person from a trained model
# is not - retraining is costly. The practical answer is to minimize and pseudonymize UP FRONT so erasure stays tractable.

- The gate lists five controls: minimize, redact before the model call, do not log raw PII, encrypt, honor erasure.
- Four pass but one — “do not log raw prompts or PII” — is unmet, so the gate **BLOCKS** the release.
- Erasure is the hardest: deleting a database row is easy, but unlearning a person from a trained model is not (retraining is costly).
- The durable fix is upstream: **minimize and pseudonymize at ingestion** so a later erasure request stays tractable — privacy by design.

#### 💡 What just happened

⚡ What just happened?Privacy is a gate: minimize, redact before the model, never log raw PII, encrypt, and honor erasure (GDPR Art. 17) - one unmet control BLOCKS the release. Tradeoff: the gate can block a launch on a logging failure, which is the point - it fails safe. That is the whole toolkit: classify PII, measure re-identification, raise k, add l-diversity, buy the DP guarantee, test for membership leakage, and gate the pipeline.

> 📦 **Info**
>
> 🚫 Two things the privacy gate does not do for youThe gate is necessary, not sufficient, and it is not legal compliance. First, **anonymization is not binary**: k-anonymity, l-diversity, and even differential privacy are dials, not switches — a determined attacker with enough auxiliary data can still narrow things down, so “anonymized” always means “anonymized against a stated threat model,” not “risk-free.” Second, **a green gate is not a lawful-basis analysis**: passing the engineering checks does not establish that you had consent, a legitimate interest, or a lawful basis to process the data at all — that is a legal determination (GDPR Article 6, a DPIA under Article 35) that sits alongside the engineering, not inside it. The engineer’s job is to make privacy measurable and the easy failures impossible; the lawful-basis and residual-risk calls belong with your privacy office and counsel.

- A five-item privacy gate with one unmet check (raw logging)
- One unmet check BLOCKS the release; unlearning is hard

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — privacy as measurable properties
> Privacy is not a checkbox but a pipeline of measurable properties from raw data to a gated release. **Classify the PII** — direct identifiers, quasi-identifiers, and the sensitive attribute (Step 1). **Measure re-identification** — records unique on their quasi-identifiers are exposed, and (ZIP + birth + sex) pins down most people (Step 2). **Raise k** — generalize and suppress until each person hides in a crowd (Step 3). **Add l-diversity** — so a homogeneous crowd does not leak the secret k was supposed to protect (Step 4). **Buy the DP guarantee** — calibrated noise that holds against an attacker with unlimited side data, spent from an epsilon budget (Step 5). **Test for membership leakage** — a model that overfits leaks who was in it, so close the loss gap (Step 6). And **gate the pipeline** — minimize, redact, never log raw PII, encrypt, and honor erasure (Step 7). Ask two questions before you ship: can I state, with a number, how re-identifiable this data is — and have I made the easy leak (raw PII in a log) impossible?

**📦 **The privacy-technique picker****

| Technique | What it protects | The guarantee | The cost |
|---|---|---|---|
| De-identification | Removes direct identifiers | Weak - quasi-IDs still re-identify | Cheap, but not real anonymity |
| k-anonymity | Hides which person (identity) | Each record shares quasi-IDs with k-1 others | Coarser data; no l-diversity hole cover |
| l-diversity | Hides the shared sensitive value | At least l distinct values per group | More generalization / suppression |
| Differential privacy | Any individual’s presence in a release | Formal: outputs change by at most e^epsilon | Noise - less accuracy at small epsilon |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now state, with numbers, how private a system is — and gate a release on it. This builds on copyright from Lesson 15.3 (the rights in the data) by protecting the *person* in the data instead. The misuse of a person’s synthetic likeness — deepfakes and synthetic identity — is Lesson 15.5. The law that turns these protections into obligations (GDPR, the EU AI Act’s data governance, the DPIA) is the regulatory landscape in Lesson 15.6, and the program that owns this privacy gate and assigns who runs it closes the module in Lesson 15.7. DP-SGD (training with a formal guarantee) and zero-retention API modes are the production frontier beyond these measures. The reference tooling is [Google’s DP library](https://github.com/google/differential-privacy) and [Presidio](https://github.com/microsoft/presidio); the governing text is [GDPR Article 17](https://gdpr-info.eu/art-17-gdpr/).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Privacy & Data Protection— Privacy as Measurable Properties**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-15_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-15.4.html` - regenerate this notebook whenever the source HTML is updated.*
